 ... grouping board + detector into a real class removes "pass `board` and
 `detector` around everywhere... " smell. Refactor below does exactly that.
 ... 

In [ ]:
# calib/charuco.py — replaces board.py + detect.py
from __future__ import annotations
from dataclasses import dataclass
import cv2
import numpy as np

from .config import BoardConfig


def _gray(image: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image


@dataclass
class Detection:
    corners: np.ndarray            # (N,1,2) float32
    ids: np.ndarray                # (N,1) int32
    image_size: tuple[int, int]    # (w, h)
    source: str = ""

    def __len__(self) -> int:
        return len(self.corners)

    def displacement_to(self, other: "Detection", min_common: int = 6) -> float | None:
        """Mean px motion of shared corners; None if too few shared. Capture-gate metric."""
        _, ia, ib = np.intersect1d(self.ids.ravel(), other.ids.ravel(), return_indices=True)
        if len(ia) < min_common:
            return None
        return float(np.linalg.norm(self.corners[ia, 0] - other.corners[ib, 0], axis=1).mean())

    def sharpness(self, image: np.ndarray) -> float:
        """Laplacian variance inside the board bbox. Focus/motion-blur gate."""
        g = _gray(image)
        (x0, y0), (x1, y1) = self.corners[:, 0].min(0).astype(int), self.corners[:, 0].max(0).astype(int) + 1
        roi = g[max(y0, 0):y1, max(x0, 0):x1]
        return float(cv2.Laplacian(roi, cv2.CV_64F).var()) if roi.size >= 100 else 0.0

    def draw(self, image: np.ndarray) -> np.ndarray:
        return cv2.aruco.drawDetectedCornersCharuco(image, self.corners, self.ids)


class CharucoBoard:
    """Board geometry + its detector, one object. The thing everyone passes around."""

    def __init__(self, cfg: BoardConfig):
        self.cfg = cfg
        dictionary = cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, cfg.dictionary))
        self.raw = cv2.aruco.CharucoBoard(
            (cfg.squares_x, cfg.squares_y), cfg.square_length_m, cfg.marker_length_m, dictionary)
        self.raw.setLegacyPattern(cfg.legacy_pattern)
        dp = cv2.aruco.DetectorParameters()
        dp.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
        cp = cv2.aruco.CharucoParameters()
        cp.tryRefineMarkers = True
        self._detector = cv2.aruco.CharucoDetector(self.raw, cp, dp)

    def detect(self, image: np.ndarray, source: str = "") -> Detection | None:
        g = _gray(image)
        corners, ids, _, _ = self._detector.detectBoard(g)
        if corners is None or len(corners) < 4:
            return None
        return Detection(corners, ids, (g.shape[1], g.shape[0]), source)

    def match(self, det: Detection) -> tuple[np.ndarray, np.ndarray]:
        """(object_pts (N,1,3) f32, image_pts (N,1,2) f32) for calibrateCamera/solvePnP."""
        obj, img = self.raw.matchImagePoints(det.corners, det.ids)
        return obj.astype(np.float32), img.astype(np.float32)

    def to_image(self, px_per_square: int = 160, margin: int = 40) -> np.ndarray:
        sx, sy = self.raw.getChessboardSize()
        return self.raw.generateImage(
            (sx * px_per_square + 2 * margin, sy * px_per_square + 2 * margin), marginSize=margin)

1. `calib/charuco.py` (The Eyes)
   The file handles everything related to actually looking at the images and
   finding the checkerboard pattern. It combines your old board configuration
   and 

In [ ]:
# calib/calibrate.py — replaces intrinsics/calibrate.py
from __future__ import annotations
from dataclasses import dataclass
import cv2
import numpy as np

from .charuco import CharucoBoard, Detection


class CalibrationError(RuntimeError):
    pass


@dataclass
class CalibrationResult:
    K: np.ndarray                  # (3,3) f64
    dist: np.ndarray               # (5,) f64 [k1,k2,p1,p2,k3] (OpenCV order)
    rms_px: float
    mse_px2: float
    max_view_rms_px: float
    per_view: list[dict]           # {source, n_corners, rms_px}
    std_intrinsics: dict           # 1-sigma from calibrateCameraExtended
    coverage_frac: float
    image_size: tuple[int, int]
    n_views: int


class IntrinsicCalibrator:
    """Zhang + LM refinement via cv2.calibrateCameraExtended (per-view errors + std devs)."""

    def __init__(self, board: CharucoBoard, min_corners: int = 8, coverage_grid: int = 4):
        self.board, self.min_corners, self.grid = board, min_corners, coverage_grid

    def calibrate(self, dets: list[Detection], image_size: tuple[int, int],
                  flags: int = 0) -> CalibrationResult:
        dets = [d for d in dets if len(d) >= self.min_corners]
        if len(dets) < 4:
            raise CalibrationError(f"need >=4 views with >={self.min_corners} corners, got {len(dets)}")
        obj, img = zip(*(self.board.match(d) for d in dets))
        criteria = (cv2.TERM_CRITERIA_MAX_ITER + cv2.TERM_CRITERIA_EPS, 200, 1e-10)
        rms, K, dist, _, _, std_i, _, view_err = cv2.calibrateCameraExtended(
            obj, img, image_size, None, None, flags=flags, criteria=criteria)
        view_err, std_i = view_err.ravel(), std_i.ravel()
        n_pts = np.array([len(d) for d in dets])
        per_view = [{"source": d.source, "n_corners": len(d), "rms_px": float(e)}
                    for d, e in zip(dets, view_err)]
        return CalibrationResult(
            K=np.asarray(K), dist=np.asarray(dist).ravel()[:5],
            rms_px=float(rms),
            mse_px2=float((view_err ** 2 * n_pts).sum() / n_pts.sum()),
            max_view_rms_px=float(view_err.max()), per_view=per_view,
            std_intrinsics={k: float(v) for k, v in zip(("fx", "fy", "cx", "cy"), std_i[:4])},
            coverage_frac=self._coverage(dets, image_size),
            image_size=image_size, n_views=len(dets))

    def cross_validate(self, dets: list[Detection], image_size) -> dict:
        """Even/odd split consistency — catches poorly-constrained params that RMS hides."""
        halves = dets[0::2], dets[1::2]
        if min(map(len, halves)) < 4:
            return {"error": "not enough views"}
        ra, rb = (self.calibrate(h, image_size) for h in halves)
        keys = {"fx": (0, 0), "fy": (1, 1), "cx": (0, 2), "cy": (1, 2)}
        out = {f"{k}_delta_pct": float(abs(ra.K[i] - rb.K[i]) / abs(rb.K[i]) * 100)
               for k, i in keys.items()}
        return out | {"k1_delta_abs": float(abs(ra.dist[0] - rb.dist[0])),
                      "rms_a": ra.rms_px, "rms_b": rb.rms_px}

    def _coverage(self, dets, image_size) -> float:
        w, h = image_size
        hit = np.zeros((self.grid, self.grid), bool)
        for d in dets:
            u, v = d.corners[:, 0, 0] / w, d.corners[:, 0, 1] / h
            hit[np.clip((v * self.grid).astype(int), 0, self.grid - 1),
                np.clip((u * self.grid).astype(int), 0, self.grid - 1)] = True
        return float(hit.mean())

In [ ]:
# calib/board_pose.py — replaces extrinsics/board_pose.py
from __future__ import annotations
import cv2
import numpy as np

from .charuco import CharucoBoard, Detection


class BoardPoseEstimator:
    """Option-C extrinsic: T_cam<-board (4x4, meters) via planar PnP (IPPE) + LM refine."""

    def __init__(self, board: CharucoBoard, min_corners: int = 6):
        self.board = board 
        self.min_corners = min_corners

    def estimate(self, det: Detection, K: np.ndarray, dist: np.ndarray) -> np.ndarray | None:
        if len(det) < self.min_corners:
            return None
        obj, img = self.board.match(det)
        ok, rvec, tvec = cv2.solvePnP(obj, img, K, dist, flags=cv2.SOLVEPNP_IPPE)
        if not ok:
            return None
        rvec, tvec = cv2.solvePnPRefineLM(obj, img, K, dist, rvec, tvec)
        T = np.eye(4)
        T[:3, :3], T[:3, 3] = cv2.Rodrigues(rvec)[0], tvec.ravel()
        return T